In [1]:
import sys
import os
import mysql.connector
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Ambil konfigurasi database dari folder utama project kalian
sys.path.append(os.path.abspath('..'))
from config import get_db_config

config = get_db_config()

# 1. Koneksi ke Database Baru (Fase Migrasi Sekarang)
db_new = mysql.connector.connect(**config['db_new'])
cursor_new = db_new.cursor(dictionary=True)

# 2. Koneksi ke Database Masa Depan (DB_FUTURE)
# Catatan: Pastikan di file config.py kalian sudah ada key 'db_future' ya!
db_future = mysql.connector.connect(**config['db_future'])
cursor_future = db_future.cursor(dictionary=True)

print(f"✅ Sukses Terhubung ke Database Baru : {config['db_new']['database']}")
print(f"🚀 Sukses Terhubung ke DB_FUTURE     : {config['db_future']['database']}")

✅ Sukses Terhubung ke Database Baru : 5
🚀 Sukses Terhubung ke DB_FUTURE     : 5


In [2]:
# # ==============================================================================
# # 1. VARIABEL UNTUK CIMUT
# # ==============================================================================
# tables_to_check = [
#     # Fase 1: Persiapan Master Data
#     "users",
#     "divisions",
#     "shift_kerja",
#     "admin_sarpras",
    
#     # Fase 2: Pendataan SDM & Wilayah Detail
#     "karyawan",
#     "keluarga_karyawan",
#     "bidang_kategori",
#     "bidang_link",
    
#     # Fase 3: Operasional, CRM & Pendaftaran (Fokus Utama)
#     "kontak_prospek",
#     "calon_siswa",
#     "calon_siswa_akademik",
#     "calon_siswa_ortu",
#     "calon_siswa_bayar",
#     "calon_siswa_jadwal",
#     "calon_siswa_kursus",
#     "calon_siswa_proses",
#     "peminjaman",
#     "pengadaan",
#     "problem",
    
#     # Fase 4: Penjadwalan & Siswa Aktif
#     "izin_karyawan",
#     "verifikasi_izin",
#     "absensi",
#     "verifikasi_absensi",
#     "karyawan_resign",

# ]


In [3]:


# ==============================================================================
# 2. VARIABEL UNTUK AFRIDA
# ==============================================================================
tables_to_check = [
    # Fase 1: Persiapan Master Data
    "kursus",
    "level",
    "sesi",
    "libur",
    "topik_diskusi",
    "kursus_level",
    "kursus_libur",
    
    # Fase 2: Pendataan SDM & Wilayah Detail
    "periode",
    "parameter_nilai",
    "kabupaten",
    "kecamatan",
    
    # Fase 3: Operasional, CRM & Pendaftaran
    "sop",
    "surat_keluar",
    "verifikasi_surat_keluar",
    "surat_tugas",
    "surat_tugas_anggota",
    "sop_kategori",
    
    # Fase 4: Penjadwalan & Siswa Aktif (Fokus Utama)
    "jadwal",
    "jadwal_hari",
    "jadwal_detail",
    "jadwal_pengajar",
    "jadwal_siswa",
    "catatan_kelas",
    "catatan_kelas_tag",
    "catatan_mingguan",
    
    # Fase 5: Penilaian & Finalisasi
    "presensi_siswa",
    "catatan_siswa",
    "followup_cs"
]


In [4]:
# # ==============================================================================
# # 3. VARIABEL UNTUK HANIF
# # ==============================================================================
# tables_to_check = [
#     # Fase 1: Persiapan Master Data
#     "roles",
#     "permissions",
#     "role_has_permissions",
#     "busdev_bidang",
#     "syarat_resign",
#     "ttd",
#     "tag_siswa_keluar",
    
#     # Fase 2: Pendataan SDM & Wilayah Detail
#     "division_user",
#     "model_has_roles",
#     "model_has_permissions",
#     "kelurahan",
    
#     # Fase 3: Operasional, CRM & Pendaftaran
#     "pengajuan_karyawan",
#     "histori_pengajuan",
#     "pelamar",
#     "pelamar_kerja",
#     "pelamar_sekolah",
#     "pelamar_kursus",
#     "progres_pelamar",
#     "rekrutmen_pelamar",
    
#     # Fase 4: Penjadwalan & Siswa Aktif (Fokus Utama)
#     "siswa",
#     "kursus_siswa",
#     "siswa_keluar",
#     "mitra",
#     "mitra_progres",
#     "kemitraan_verifikator",
#     "siswa_mitra",
#     "siswa_mitra_keluar",
    
#     # Fase 5: Penilaian & Finalisasi (Fokus Utama)
#     "rapor_format",
#     "rapor_format_sub",
#     "rapor_format_formula",
#     "rapor_format_formula_sub",
#     "rapor_level_config",
#     "rapor_sub_level",
#     "rapor_siswa",
#     "rapor_siswa_file",
#     "rapor_lacak"
# ]

In [5]:
import pandas as pd
import numpy as np

print("================================================================================")
print(" 🚀 BLUEPRINT INSPECTOR: SCAN SPESIFIK TABEL DI DB_FUTURE 🚀 ")
print("================================================================================")

print(f"Memulai inspeksi untuk {len(tables_to_check)} tabel spesifik di {config['db_future']['database']}...\n")

# --- TAHAP INSPEKSI & RENDER VISUAL ---
for table in tables_to_check:
    try:
        # 2. Ambil sampel data dari tabel di DB_FUTURE
        query_data = f"SELECT * FROM `{table}`"
        df_real_data = pd.read_sql(query_data, db_future)
        
        # 3. Tarik Struktur Fisik Kolom murni dari DB_FUTURE
        query_struct = f"""
        SELECT 
            c.COLUMN_NAME AS 'Nama Kolom',
            c.COLUMN_TYPE AS 'Tipe Data MySQL',
            CONCAT(
                CASE WHEN c.IS_NULLABLE = 'YES' THEN '✅ NULL (Boleh Kosong)' ELSE '🛑 NOT NULL (Wajib Isi)' END,
                CASE WHEN c.EXTRA = 'auto_increment' THEN '\\n🚀 AUTO_INCREMENT' ELSE '' END
            ) AS 'Aturan Nullability & Increment',
            CASE 
                WHEN c.COLUMN_KEY = 'PRI' THEN '🔑 PRIMARY KEY (PK)'
                WHEN c.COLUMN_KEY = 'MUL' AND k.REFERENCED_TABLE_NAME IS NOT NULL THEN '🔗 FOREIGN KEY (FK)'
                WHEN c.COLUMN_KEY = 'MUL' THEN 'INDEX'
                ELSE '-'
            END AS 'Status Kunci',
            CASE 
                WHEN k.REFERENCED_TABLE_NAME IS NOT NULL THEN CONCAT(k.REFERENCED_TABLE_NAME, ' (', k.REFERENCED_COLUMN_NAME, ')')
                ELSE '-'
            END AS 'Rujukan Induk (FK Origin)',
            CASE WHEN c.DATA_TYPE = 'enum' THEN REPLACE(REPLACE(REPLACE(c.COLUMN_TYPE, 'enum(', ''), ')', ''), "'", "") ELSE '-' END AS 'Daftar Pilihan ENUM'
        FROM INFORMATION_SCHEMA.COLUMNS c
        LEFT JOIN INFORMATION_SCHEMA.KEY_COLUMN_USAGE k 
            ON c.TABLE_SCHEMA = k.TABLE_SCHEMA AND c.TABLE_NAME = k.TABLE_NAME AND c.COLUMN_NAME = k.COLUMN_NAME
        WHERE c.TABLE_SCHEMA = '{config['db_future']['database']}' AND c.TABLE_NAME = '{table}'
        ORDER BY c.ORDINAL_POSITION;
        """
        df_struct = pd.read_sql(query_struct, db_future).drop_duplicates(subset=['Nama Kolom'], keep='first').reset_index(drop=True)
        
        # Cek jika tabel ternyata tidak ada (typo atau belum dibuat)
        if df_struct.empty:
            print(f"\n================================================================================")
            print(f"❌ TABEL: {table.upper()} (TIDAK DITEMUKAN DI DB_FUTURE)")
            print(f"================================================================================")
            continue

        # 4. Lacak tabel anak yang menjadikan PK tabel ini sebagai FK mereka
        pk_referenced_list = []
        for idx, row_skri in df_struct.iterrows():
            if row_skri['Status Kunci'] == '🔑 PRIMARY KEY (PK)':
                lookup_fk_query = f"""
                SELECT CONCAT(TABLE_NAME, ' (', COLUMN_NAME, ')') as relasi
                FROM INFORMATION_SCHEMA.KEY_COLUMN_USAGE
                WHERE REFERENCED_TABLE_SCHEMA = '{config['db_future']['database']}'
                  AND REFERENCED_TABLE_NAME = '{table}'
                  AND REFERENCED_COLUMN_NAME = '{row_skri['Nama Kolom']}';
                """
                df_relasi_luar = pd.read_sql(lookup_fk_query, db_future)
                if not df_relasi_luar.empty:
                    pk_referenced_list.append("\n".join(df_relasi_luar['relasi'].tolist()))
                else:
                    pk_referenced_list.append("-")
            else:
                pk_referenced_list.append("-")
        df_struct['Tabel Yang nge-FK ke Sini'] = pk_referenced_list

        # ---------------------------------------------------------
        # TAMPILAN RENDER PAPAN INFO
        # ---------------------------------------------------------
        print(f"\n================================================================================")
        print(f"📦 TABEL: {table.upper()}")
        print(f"================================================================================")
        
        print(f"📋 1. Karakteristik Wadah Pandas Dataframe (.info()):")
        df_real_data.info()
        print("\n" + "-"*60)
        
        print(f"🔍 2. Arsitektur Fisik Kolom pada DB_FUTURE:")
        display(df_struct.style.set_properties(**{'white-space': 'pre-wrap', 'text-align': 'left'}))
        print("\n" + "-"*60)
        
        print(f"📸 3. Sampel Isi Data Real di DB_FUTURE (Menampilkan Max 5 Baris Teratas):")
        # Menggunakan .head(5) agar visualisasi layar tidak terlalu panjang
        display(df_real_data if not df_real_data.empty else df_real_data)
        print("\n" + "="*80)
            
    except Exception as e:
        print(f"❌ Gagal menganalisis tabel `{table}`: {e}")

print("\n🎉 SCANNING SELESAI! 🎉")

 🚀 BLUEPRINT INSPECTOR: SCAN SPESIFIK TABEL DI DB_FUTURE 🚀 
Memulai inspeksi untuk 28 tabel spesifik di 5...




📦 TABEL: KURSUS
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   id_kursus     20 non-null     object
 1   nama_kursus   20 non-null     object
 2   deskripsi     20 non-null     object
 3   tipe_kursus   20 non-null     object
 4   status_arsip  20 non-null     int64 
dtypes: int64(1), object(4)
memory usage: 932.0+ bytes

------------------------------------------------------------
🔍 2. Arsitektur Fisik Kolom pada DB_FUTURE:


,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK ke Sini
0,id_kursus,varchar(15),🛑 NOT NULL (Wajib Isi),🔑 PRIMARY KEY (PK),-,-,calon_siswa_akademik (id_kursus) calon_siswa_form_programs (id_kursus) jadwal (id_kursus) kursus_level (id_kursus) kursus_libur (id_kursus) kursus_siswa (id_kursus) periode (id_kursus) rapor_format (id_kursus) rapor_level_config (id_kursus) rapor_setting_kursus (id_kursus) siswa_keluar (id_kursus) siswa_keluar_feedbacks (id_kursus)
1,nama_kursus,varchar(150),🛑 NOT NULL (Wajib Isi),-,-,-,-
2,deskripsi,text,🛑 NOT NULL (Wajib Isi),-,-,-,-
3,tipe_kursus,"enum('B2C','B2B')",🛑 NOT NULL (Wajib Isi),-,-,"B2C,B2B",-
4,status_arsip,tinyint(1),🛑 NOT NULL (Wajib Isi),-,-,-,-



------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_FUTURE (Menampilkan Max 5 Baris Teratas):


,id_kursus,nama_kursus,deskripsi,tipe_kursus,status_arsip
0,K00001,LEAP - General English,"GE, Balloons, Gogo, SO, Winner",B2C,0
1,K00002,LEAP - Coding Class,Coding Class Regular,B2C,0
2,K00003,LEAP - Leap Literacy Club,LLC,B2C,0
3,K00004,LEAP - Conversation Class,Conversation Class for Adults,B2C,0
4,K00005,LEAP - Aplikasi Perkantoran,"All In, Private,",B2C,0
5,K00006,Kemitraan - B2B Supervisi Guru,Nurul Faizah,B2B,0
6,K00007,Kemitraan - B2B Business English,"Delta Jaya, Hartono, PT HCA",B2B,0
7,K00009,Kemitraan - B2C Talentvis,Talentvis,B2B,0
8,K00010,LEAP - General English 2024,NEW CURRICULUM 2024,B2C,0
9,K00011,Kemitraan - B2B Language Upskilling Program,Nurul Faizah Agt 24 - Mei 25,B2B,0


...



📦 TABEL: FOLLOWUP_CS
📋 1. Karakteristik Wadah Pandas Dataframe (.info()):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 0 entries
Data columns (total 6 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id_followup_cs          0 non-null      object
 1   id_cs                   0 non-null      object
 2   tanggal_followup        0 non-null      object
 3   id_user                 0 non-null      object
 4   kesimpulan_followup_cs  0 non-null      object
 5   status_followup         0 non-null      object
dtypes: object(6)
memory usage: 132.0+ bytes

------------------------------------------------------------
🔍 2. Arsitektur Fisik Kolom pada DB_FUTURE:


,Nama Kolom,Tipe Data MySQL,Aturan Nullability & Increment,Status Kunci,Rujukan Induk (FK Origin),Daftar Pilihan ENUM,Tabel Yang nge-FK ke Sini
0,id_followup_cs,bigint(20) unsigned,🛑 NOT NULL (Wajib Isi) 🚀 AUTO_INCREMENT,🔑 PRIMARY KEY (PK),-,-,-
1,id_cs,bigint(20) unsigned,✅ NULL (Boleh Kosong),🔗 FOREIGN KEY (FK),catatan_siswa (id_cs),-,-
2,tanggal_followup,timestamp,🛑 NOT NULL (Wajib Isi),-,-,-,-
3,id_user,varchar(20),✅ NULL (Boleh Kosong),🔗 FOREIGN KEY (FK),users (id_user),-,-
4,kesimpulan_followup_cs,text,🛑 NOT NULL (Wajib Isi),-,-,-,-
5,status_followup,"enum('NEED FURTHER OBSERVATION','CASE CLOSED')",🛑 NOT NULL (Wajib Isi),-,-,"NEED FURTHER OBSERVATION,CASE CLOSED",-



------------------------------------------------------------
📸 3. Sampel Isi Data Real di DB_FUTURE (Menampilkan Max 5 Baris Teratas):


,id_followup_cs,id_cs,tanggal_followup,id_user,kesimpulan_followup_cs,status_followup




🎉 SCANNING SELESAI! 🎉
